# Comment classifier: XLM-R fine-tune (P2, prd.md Member C deliverable 4)

Run on **Colab or Kaggle with a GPU** (CLAUDE.md hard constraint 5). It uses the **same split** as
`ml/comments/train.py`, so its score is comparable with the CPU model and both baselines.

1. Upload your Label Studio export as `comments.json` (never commit it).
2. Run all cells.
3. Download `xlmr-comments.zip` and unzip it on the AI worker. Set `COMMENT_MODEL_PATH` to the folder.

Library calls follow the transformers and optimum docs. Check them against the versions Colab installs on the first run.

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "optimum[onnxruntime]>=1.21" scikit-learn
!git clone -q https://github.com/Abubakar-Zahir/FYP.git repo
import sys

sys.path.insert(0, "repo")

In [ ]:
import json

from ml.comments.dataset import from_label_studio, split_by_group
from ml.comments.labels import CommentLabel
from ml.comments.normalise import normalise

items = from_label_studio(json.load(open("comments.json", encoding="utf-8")))
train, test = split_by_group(items)
LABELS = [label.value for label in CommentLabel]
print(len(train), "train,", len(test), "test")

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(13)
BASE = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(BASE)


def to_dataset(rows):
    data = Dataset.from_dict(
        {"text": [normalise(r.text) for r in rows], "label": [LABELS.index(r.label.value) for r in rows]}
    )
    return data.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=128), batched=True)


train_data, test_data = to_dataset(train), to_dataset(test)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(LABELS))
args = TrainingArguments(
    "xlmr-out",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    seed=13,
    report_to=[],
    save_strategy="no",
)
trainer = Trainer(model=model, args=args, train_dataset=train_data, tokenizer=tokenizer)
trainer.train()

In [ ]:
import numpy as np

from ml.comments.evaluate import evaluate

logits = trainer.predict(test_data).predictions
predicted = [LABELS[i] for i in np.argmax(logits, axis=1)]
report = evaluate("xlmr-base-v1", [r.label.value for r in test], predicted)
print(report.model_dump_json(indent=2))
json.dump(report.model_dump(), open("xlmr_report.json", "w"), indent=2)

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

trainer.save_model("xlmr-final")
tokenizer.save_pretrained("xlmr-final")
onnx_model = ORTModelForSequenceClassification.from_pretrained("xlmr-final", export=True)
onnx_model.save_pretrained("xlmr-comments")
tokenizer.save_pretrained("xlmr-comments")  # writes tokenizer.json
json.dump(LABELS, open("xlmr-comments/labels.json", "w"))
!cp xlmr_report.json xlmr-comments/ && zip -qr xlmr-comments.zip xlmr-comments

In [ ]:
# Same inference path the worker uses, to check the export before downloading.
from pathlib import Path

from ml.comments.onnx_model import XlmrOnnxClassifier

print(XlmrOnnxClassifier(Path("xlmr-comments")).predict(["Ye kahan hai?", "Awaz bohat kharab hai"]))